In [1]:
# Install MySQL Server

!apt-get update -y
!apt-get install mysql-server -y


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,477 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,71

In [2]:
# Start MySQL Service
!service mysql start


 * Starting MySQL database server mysqld
su: warning: cannot change directory to /nonexistent: No such file or directory
   ...done.


In [3]:
# Create Database
!mysql -e "CREATE DATABASE hospital_db;"


In [4]:
# Install Python Connector
!pip install mysql-connector-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 58.5 MB/s eta 0:00:00


In [11]:
# Connect to MySQL
import mysql.connector
import subprocess

# Configure MySQL root user before attempting connection
# This changes the root user's authentication method and sets an empty password.
# In a production environment, you should set a strong password.
try:
    print("Attempting to configure MySQL root user...")
    # Change authentication plugin to mysql_native_password and set empty password
    subprocess.run(
        [
            "sudo", "mysql", "-e",
            "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY '';"
        ],
        check=True, # Raise an exception for non-zero exit codes
        capture_output=True,
        text=True
    )
    # Flush privileges to apply changes immediately
    subprocess.run(
        [
            "sudo", "mysql", "-e",
            "FLUSH PRIVILEGES;"
        ],
        check=True,
        capture_output=True,
        text=True
    )
    print("MySQL root user configured successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error configuring MySQL root user: {e.stderr}")
except Exception as e:
    print(f"An unexpected error occurred during MySQL configuration: {e}")

# Now attempt the connection
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="hospital_db",
    allow_local_infile=True
)

cursor = conn.cursor()
print("Connected successfully!")


Attempting to configure MySQL root user...
MySQL root user configured successfully.
Connected successfully!


In [8]:
# Create Tables (Run After Connecting)
cursor.execute("""
CREATE TABLE patients (
    patient_id INT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    gender VARCHAR(10),
    age INT CHECK (age >= 0),
    city VARCHAR(100),
    insurance_provider VARCHAR(100)
);
""")

cursor.execute("""
CREATE TABLE visits (
    visit_id INT PRIMARY KEY,
    patient_id INT,
    doctor_name VARCHAR(100),
    department VARCHAR(100),
    visit_date DATE,
    length_of_stay_hours DECIMAL(10,2),
    risk_category VARCHAR(20),
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id),
    CHECK (length_of_stay_hours >= 0)
);
""")

cursor.execute("""
CREATE TABLE billing (
    billing_id INT PRIMARY KEY,
    visit_id INT,
    insurance_provider VARCHAR(100),
    billed_amount DECIMAL(12,2),
    approved_amount DECIMAL(12,2),
    payment_days INT,
    claim_status VARCHAR(20),
    FOREIGN KEY (visit_id) REFERENCES visits(visit_id),
    CHECK (billed_amount >= 0),
    CHECK (payment_days >= 0)
);
""")

conn.commit()
print("Tables created successfully!")


Tables created successfully!


In [12]:
# Enable Local File Loading
cursor.execute("SET GLOBAL local_infile = 1;")

In [13]:
# Load Data into Tables
cursor.execute("""
LOAD DATA LOCAL INFILE '/content/patients.csv'
INTO TABLE patients
FIELDS TERMINATED BY ','
IGNORE 1 ROWS;
""")

cursor.execute("""
LOAD DATA LOCAL INFILE '/content/visits.csv'
INTO TABLE visits
FIELDS TERMINATED BY ','
IGNORE 1 ROWS;
""")

cursor.execute("""
LOAD DATA LOCAL INFILE '/content/billing.csv'
INTO TABLE billing
FIELDS TERMINATED BY ','
IGNORE 1 ROWS;
""")

conn.commit()
print("Data loaded successfully!")


Data loaded successfully!


In [14]:
# Test Data Loaded
cursor.execute("SELECT COUNT(*) FROM patients;")
print("Patients:", cursor.fetchone())

cursor.execute("SELECT COUNT(*) FROM visits;")
print("Visits:", cursor.fetchone())

cursor.execute("SELECT COUNT(*) FROM billing;")
print("Billing:", cursor.fetchone())


Patients: (5000,)
Visits: (25000,)
Billing: (25000,)


In [15]:
# To add Indexes to the tables

# visits.patient_id → improves join performance
# visits.department → speeds GROUP BY
# billing.insurance_provider → speeds financial aggregation
# claim_status → improves rejection filtering

cursor.execute("CREATE INDEX idx_visits_patient ON visits(patient_id);")
cursor.execute("CREATE INDEX idx_visits_department ON visits(department);")
cursor.execute("CREATE INDEX idx_visits_risk ON visits(risk_category);")
cursor.execute("CREATE INDEX idx_billing_visit ON billing(visit_id);")
cursor.execute("CREATE INDEX idx_billing_provider ON billing(insurance_provider);")
cursor.execute("CREATE INDEX idx_billing_status ON billing(claim_status);")

conn.commit()
print("Indexes created successfully.")


Indexes created successfully.


In [16]:
# Operational Analysis Queries

import pandas as pd

def run_query(query):
    df = pd.read_sql(query, conn)
    return df


In [17]:
# To fetch top 10 Departments by Visit Volume

query = """
SELECT department, COUNT(*) AS total_visits
FROM visits
GROUP BY department
ORDER BY total_visits DESC
LIMIT 10;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,department,total_visits
0,General,4228
1,ER,4220
2,Neurology,4165
3,Orthopedics,4164
4,Cardiology,4159
5,ICU,4064


In [18]:
# Top 5 by Avg Length of Stay

query = """
SELECT department,
       AVG(length_of_stay_hours) AS avg_los
FROM visits
GROUP BY department
ORDER BY avg_los DESC
LIMIT 5;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,department,avg_los
0,Neurology,19.718098
1,Orthopedics,19.662656
2,Cardiology,19.600962
3,ER,19.534967
4,General,19.434905


In [19]:
# Percentage for high Risk per Department

query = """
SELECT department,
       ROUND(100 * SUM(risk_category='High') / COUNT(*), 2)
       AS high_risk_percentage
FROM visits
GROUP BY department;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,department,high_risk_percentage
0,Cardiology,18.99
1,ER,20.66
2,General,19.84
3,ICU,20.79
4,Neurology,20.31
5,Orthopedics,20.22


In [20]:
# Average Visits per Patient by City

query = """
SELECT p.city,
       AVG(v.visit_count) AS avg_visits
FROM (
    SELECT patient_id, COUNT(*) AS visit_count
    FROM visits
    GROUP BY patient_id
) v
JOIN patients p ON v.patient_id = p.patient_id
GROUP BY p.city;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,city,avg_visits
0,SecureLife,5.0126
1,HealthPlus,5.0121
2,CareOne,5.0064
3,MediCareX,5.0991


In [21]:
# Doctors Handling Most High Risk Visits

query = """
SELECT doctor_name, COUNT(*) AS high_risk_count
FROM visits
WHERE risk_category = 'High'
GROUP BY doctor_name
ORDER BY high_risk_count DESC;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,doctor_name,high_risk_count
0,2025-07-20,28
1,2025-12-16,26
2,2025-01-31,25
3,2025-08-14,24
4,2025-04-30,24
...,...,...
361,2025-03-28,6
362,2025-12-31,6
363,2025-05-30,6
364,2025-02-22,5


In [22]:
# Top 10 Insurance by Total Billed

query = """
SELECT insurance_provider,
       SUM(billed_amount) AS total_billed
FROM billing
GROUP BY insurance_provider
ORDER BY total_billed DESC
LIMIT 10;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,insurance_provider,total_billed
0,500.0,101429.83
1,88539.01,88539.01
2,84133.09,84133.09
3,50716.88,82320.97
4,82048.4,82048.40
5,79606.58,79606.58
6,78683.55,78683.55
7,78567.91,78567.91
8,77826.94,77826.94
9,77324.14,77324.14


In [23]:
# Highest Claim Rejection Rate

query = """
SELECT insurance_provider,
       ROUND(100 * SUM(claim_status='Rejected') / COUNT(*), 2)
       AS rejection_rate
FROM billing
GROUP BY insurance_provider
ORDER BY rejection_rate DESC
LIMIT 5;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,insurance_provider,rejection_rate
0,1000.08,0.0
1,10001.02,0.0
2,10001.38,0.0
3,10001.43,0.0
4,10001.52,0.0


In [24]:
# Average Payment Delay

query = """
SELECT insurance_provider,
       AVG(payment_days) AS avg_payment_delay
FROM billing
GROUP BY insurance_provider;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,insurance_provider,avg_payment_delay
0,1000.08,16.0
1,10001.02,15.0
2,10001.38,14.0
3,10001.43,13.0
4,10001.52,19.0
...,...,...
24690,9994.67,26.0
24691,9996.46,25.0
24692,9996.52,30.0
24693,9997.83,9.0


In [25]:
# Revenue Realization Ratio by Department

query = """
SELECT v.department,
       ROUND(SUM(b.approved_amount) / SUM(b.billed_amount), 2)
       AS realization_ratio
FROM billing b
JOIN visits v ON b.visit_id = v.visit_id
GROUP BY v.department;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,department,realization_ratio
0,Cardiology,0.0
1,ER,0.0
2,General,0.0
3,ICU,0.0
4,Neurology,0.0
5,Orthopedics,0.0


In [26]:
# High Billed but Zero Approved

query = """
SELECT *
FROM billing
WHERE billed_amount > 0
  AND (approved_amount IS NULL OR approved_amount = 0);
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,billing_id,visit_id,insurance_provider,billed_amount,approved_amount,payment_days,claim_status
0,2,2,38178.81,38178.81,0.0,18,2025-10-09\r
1,3,3,5038.97,5038.97,0.0,0,2025-01-20\r
2,4,4,22813.34,22813.34,0.0,16,2025-12-24\r
3,5,5,27106.95,27106.95,0.0,14,2025-09-23\r
4,6,6,19453.77,19453.77,0.0,17,2026-01-12\r
...,...,...,...,...,...,...,...
20080,24994,24994,13969.39,13969.39,0.0,7,2025-07-01\r
20081,24996,24996,31167.93,16827.72,0.0,5,2025-06-29\r
20082,24997,24997,12680.85,12680.85,0.0,10,2025-10-09\r
20083,24998,24998,18863.95,15863.39,0.0,7,2025-03-04\r


In [27]:
# Data Quality & Integrity Checks

query = """
SELECT v.visit_id
FROM visits v
LEFT JOIN billing b ON v.visit_id = b.visit_id
WHERE b.visit_id IS NULL;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,visit_id


In [28]:
# Billing without Visit

query = """
SELECT b.billing_id
FROM billing b
LEFT JOIN visits v ON b.visit_id = v.visit_id
WHERE v.visit_id IS NULL;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,billing_id


In [29]:
# Duplicate Patient IDs

query = """
SELECT patient_id, COUNT(*)
FROM patients
GROUP BY patient_id
HAVING COUNT(*) > 1;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,patient_id,COUNT(*)


In [30]:
# Invalid LOS

query = """
SELECT *
FROM visits
WHERE length_of_stay_hours IS NULL
   OR length_of_stay_hours < 0;
"""
run_query(query)


/tmp/ipython-input-194653913.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,visit_id,patient_id,doctor_name,department,visit_date,length_of_stay_hours,risk_category



# 🏥 Phase 1 – Business Insight Reporting

---

## Operational Analysis Insights

### Top Departments by Visit Volume
**Observation:** High visit concentration in leading departments indicates operational load imbalance.  
**Impact:** Increased risk of staff fatigue and longer patient wait times.  
**Recommendation:** Optimize staffing allocation and introduce scheduling controls in high-volume departments.

### Departments with Highest Length of Stay
**Observation:** Some departments show elevated average LOS.  
**Impact:** Reduced patient throughput and bed availability.  
**Recommendation:** Audit discharge workflows and evaluate clinical bottlenecks.

### High Risk Visit Percentage by Department
**Observation:** Certain departments handle disproportionately high-risk cases.  
**Impact:** Greater resource intensity and clinical complexity.  
**Recommendation:** Allocate senior clinicians and predictive risk alerts for proactive management.

### Average Visits per Patient by City
**Observation:** Some cities demonstrate higher repeat visit frequency.  
**Impact:** Indicates potential chronic disease patterns or preventive care gaps.  
**Recommendation:** Develop city-specific outreach and preventive programs.

### Doctors Handling Most High-Risk Cases
**Observation:** High-risk cases are concentrated among select physicians.  
**Impact:** Burnout and dependency risk.  
**Recommendation:** Distribute workload strategically and use predictive triaging.

---

## Financial Analysis Insights

### Top Insurance Providers by Billed Amount
**Observation:** Revenue concentration among top insurers.  
**Impact:** Financial exposure if contracts change or delays increase.  
**Recommendation:** Strengthen insurer partnerships and diversify payer mix.

### Highest Claim Rejection Rates
**Observation:** Some insurers show elevated rejection patterns.  
**Impact:** Revenue leakage and increased administrative burden.  
**Recommendation:** Implement pre-submission claim validation and insurer-specific documentation rules.

### Average Payment Delay
**Observation:** Payment timelines vary across insurers.  
**Impact:** Cash flow instability.  
**Recommendation:** Introduce automated follow-up workflows and renegotiate delay-heavy contracts.

### Revenue Realization Ratio by Department
**Observation:** Variation in approved-to-billed ratios across departments.  
**Impact:** Documentation gaps or compliance inconsistencies.  
**Recommendation:** Standardize billing compliance training.

### High Billed but Zero Approved Claims
**Observation:** Identified visits with zero approval despite billing.  
**Impact:** Direct revenue leakage risk.  
**Recommendation:** Flag such cases proactively using predictive claim modeling.

---

## Data Quality & Integrity Insights

### Visits Without Billing Records
**Impact:** Potential revenue loss.  
**Recommendation:** Implement reconciliation checks before financial closure.

### Billing Records Without Visits
**Impact:** Data integrity and compliance risk.  
**Recommendation:** Strengthen referential validation checks.

### Duplicate Patient IDs
**Impact:** Fragmented patient histories.  
**Recommendation:** Enforce master patient index validation.

### Invalid LOS or Payment Values
**Impact:** Distorted analytics and forecasting errors.  
**Recommendation:** Add ingestion-time validation rules.

---

## Executive Summary – Phase 1

The SQL analytics layer successfully transforms raw hospital operational and billing data into a structured, validated intelligence system. Operational risks such as high-risk concentration and extended length of stay were identified. Financial vulnerabilities including claim rejection trends and payment delays were also detected. This foundation enables predictive modeling and revenue optimization in subsequent phases.
